# Week 77, Day 1: Flask

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w077d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('miniflask.py', 'w', encoding='utf-8') as f:
    f.write('"""A small subset of Flask that runs in the page (Week 77).\n\nThe names and behaviour follow Flask\'s own, so code written against this module also runs with\nthe real Flask (pip install flask): Flask, request, g, jsonify, abort, redirect, make_response,\nurl_for, Blueprint, and app.test_client(). It\'s a teaching copy, not the real thing: no templates,\nsessions, streaming or debugger.\n"""\nimport io\nimport json\nimport re\nfrom contextvars import ContextVar\nfrom http import HTTPStatus\nfrom urllib.parse import parse_qs, urlencode, urlsplit\n\n_current = ContextVar("request")\n\n\nclass HTTPException(Exception):\n    def __init__(self, code, description=None):\n        self.code = code\n        self.name = HTTPStatus(code).phrase\n        self.description = description or self.name\n        super().__init__(f"{code} {self.name}: {self.description}")\n\n\ndef abort(code, description=None):\n    raise HTTPException(code, description)\n\n\nclass Headers:\n    """Case-insensitive headers."""\n    def __init__(self, items=()):\n        self._items = [(k, str(v)) for k, v in (items.items() if isinstance(items, dict) else items)]\n\n    def get(self, name, default=None):\n        for k, v in self._items:\n            if k.lower() == name.lower():\n                return v\n        return default\n\n    def __getitem__(self, name):\n        value = self.get(name)\n        if value is None:\n            raise KeyError(name)\n        return value\n\n    def __contains__(self, name):\n        return self.get(name) is not None\n\n    def __setitem__(self, name, value):\n        self._items = [(k, v) for k, v in self._items if k.lower() != name.lower()] + [(name, str(value))]\n\n    def items(self):\n        return list(self._items)\n\n\nclass Args(dict):\n    """Query or form values: .get(name) gives the first value; .getlist(name) all of them."""\n    def __init__(self, pairs):\n        super().__init__()\n        self._lists = {}\n        for k, v in pairs:\n            self._lists.setdefault(k, []).append(v)\n            dict.setdefault(self, k, v)\n\n    def get(self, key, default=None, type=None):\n        if key not in self:\n            return default\n        value = self[key]\n        if type is not None:\n            try:\n                return type(value)\n            except (TypeError, ValueError):\n                return default\n        return value\n\n    def getlist(self, key):\n        return list(self._lists.get(key, []))\n\n\nclass Request:\n    def __init__(self, environ):\n        self.environ = environ\n        self.method = environ["REQUEST_METHOD"].upper()\n        self.path = environ.get("PATH_INFO", "/") or "/"\n        self.args = Args(_pairs(environ.get("QUERY_STRING", "")))\n        items = [(k[5:].replace("_", "-").title(), v) for k, v in environ.items() if k.startswith("HTTP_")]\n        if environ.get("CONTENT_TYPE"):\n            items.append(("Content-Type", environ["CONTENT_TYPE"]))\n        self.headers = Headers(items)\n        length = int(environ.get("CONTENT_LENGTH") or 0)\n        self.data = environ["wsgi.input"].read(length) if length else b""\n        self.content_type = environ.get("CONTENT_TYPE", "")\n        self.form = Args(_pairs(self.data.decode("utf-8"))) if self.content_type.startswith("application/x-www-form-urlencoded") else Args([])\n        self.view_args = {}\n        cookie = environ.get("HTTP_COOKIE", "")\n        self.cookies = dict(p.strip().split("=", 1) for p in cookie.split(";") if "=" in p)\n\n    @property\n    def is_json(self):\n        return self.content_type.split(";")[0].strip() == "application/json"\n\n    def get_json(self, silent=False, force=False):\n        if not (self.is_json or force):\n            if silent:\n                return None\n            abort(415, "expected a JSON body (Content-Type: application/json)")\n        try:\n            return json.loads(self.data or b"null")\n        except ValueError:\n            if silent:\n                return None\n            abort(400, "the body isn\'t valid JSON")\n\n    @property\n    def json(self):\n        return self.get_json()\n\n\ndef _pairs(text):\n    return [(k, v) for k, vs in parse_qs(text, keep_blank_values=True).items() for v in vs]\n\n\nclass _Proxy:\n    def __init__(self, getter):\n        object.__setattr__(self, "_getter", getter)\n\n    def __getattr__(self, name):\n        return getattr(self._getter(), name)\n\n    def __setattr__(self, name, value):\n        setattr(self._getter(), name, value)\n\n\ndef _request():\n    try:\n        return _current.get()[0]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\ndef _g():\n    try:\n        return _current.get()[1]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\nclass _G:\n    def get(self, name, default=None):\n        return self.__dict__.get(name, default)\n\n\nrequest = _Proxy(_request)\ng = _Proxy(_g)\n\n\nclass Response:\n    def __init__(self, response=b"", status=200, headers=None, mimetype=None, content_type=None):\n        if isinstance(response, str):\n            response = response.encode("utf-8")\n        self.data = response\n        self.status_code = status\n        self.headers = Headers(headers or {})\n        if content_type or mimetype or "Content-Type" not in self.headers:\n            ctype = content_type or (f"{mimetype}; charset=utf-8" if mimetype and mimetype.startswith("text/") else mimetype) or "text/html; charset=utf-8"\n            self.headers["Content-Type"] = ctype\n\n    @property\n    def status(self):\n        return f"{self.status_code} {HTTPStatus(self.status_code).phrase}"\n\n    @property\n    def text(self):\n        return self.data.decode("utf-8")\n\n    def get_json(self):\n        return json.loads(self.data)\n\n    @property\n    def json(self):\n        return self.get_json()\n\n    def set_cookie(self, key, value="", max_age=None, path="/", secure=False, httponly=False, samesite=None):\n        parts = [f"{key}={value}"]\n        if max_age is not None:\n            parts.append(f"Max-Age={max_age}")\n        parts.append(f"Path={path}")\n        if secure:\n            parts.append("Secure")\n        if httponly:\n            parts.append("HttpOnly")\n        if samesite:\n            parts.append(f"SameSite={samesite}")\n        self.headers._items.append(("Set-Cookie", "; ".join(parts)))\n\n\ndef jsonify(*args, **kwargs):\n    data = args[0] if len(args) == 1 else (list(args) if args else kwargs)\n    return Response(json.dumps(data), headers={"Content-Type": "application/json"})\n\n\ndef redirect(location, code=302):\n    return Response(b"", code, {"Location": location})\n\n\ndef make_response(*args):\n    rv = args[0] if len(args) == 1 else args\n    return _App._to_response(rv)\n\n\ndef url_for(endpoint, **values):\n    return _current.get()[2].url_for(endpoint, **values)\n\n\n_CONVERTERS = {"int": (r"\\d+", int), "float": (r"\\d+(?:\\.\\d+)?", float), "path": (r".+", str), "string": (r"[^/]+", str)}\n\n\nclass _Rule:\n    def __init__(self, rule, endpoint, view, methods):\n        self.rule, self.endpoint, self.view = rule, endpoint, view\n        self.methods = {m.upper() for m in methods} | ({"HEAD"} if "GET" in {m.upper() for m in methods} else set())\n        regex, self.converters = "", {}\n        for piece in re.split(r"(<[^>]+>)", rule):\n            m = re.fullmatch(r"<(?:(\\w+):)?(\\w+)>", piece)\n            if m:\n                kind, name = m.group(1) or "string", m.group(2)\n                pattern, convert = _CONVERTERS[kind]\n                regex += f"(?P<{name}>{pattern})"\n                self.converters[name] = convert\n            else:\n                regex += re.escape(piece)\n        self.regex = re.compile(regex)\n\n    def match(self, path):\n        m = self.regex.fullmatch(path)\n        if not m:\n            return None\n        return {k: self.converters[k](v) for k, v in m.groupdict().items()}\n\n    def build(self, values):\n        def fill(m):\n            return str(values.pop(m.group(2)))\n        path = re.sub(r"<(?:(\\w+):)?(\\w+)>", fill, self.rule)\n        return path + (f"?{urlencode(values)}" if values else "")\n\n\nclass _Setup:\n    def route(self, rule, methods=("GET",), endpoint=None):\n        def decorate(view):\n            self._add(rule, endpoint or view.__name__, view, methods)\n            return view\n        return decorate\n\n    def get(self, rule, **kw):\n        return self.route(rule, methods=["GET"], **kw)\n\n    def post(self, rule, **kw):\n        return self.route(rule, methods=["POST"], **kw)\n\n    def put(self, rule, **kw):\n        return self.route(rule, methods=["PUT"], **kw)\n\n    def patch(self, rule, **kw):\n        return self.route(rule, methods=["PATCH"], **kw)\n\n    def delete(self, rule, **kw):\n        return self.route(rule, methods=["DELETE"], **kw)\n\n\nclass Blueprint(_Setup):\n    def __init__(self, name, import_name, url_prefix=""):\n        self.name, self.url_prefix = name, url_prefix\n        self.deferred = []\n\n    def _add(self, rule, endpoint, view, methods):\n        self.deferred.append((rule, endpoint, view, methods))\n\n\nclass _App(_Setup):\n    def __init__(self, import_name):\n        self.import_name = import_name\n        self.config = {"TESTING": False}\n        self.rules = []\n        self.before_funcs, self.after_funcs, self.error_handlers = [], [], {}\n\n    def _add(self, rule, endpoint, view, methods):\n        self.rules.append(_Rule(rule, endpoint, view, methods))\n\n    def register_blueprint(self, bp, url_prefix=None):\n        prefix = url_prefix if url_prefix is not None else bp.url_prefix\n        for rule, endpoint, view, methods in bp.deferred:\n            self._add(prefix + rule, f"{bp.name}.{endpoint}", view, methods)\n\n    def before_request(self, f):\n        self.before_funcs.append(f)\n        return f\n\n    def after_request(self, f):\n        self.after_funcs.append(f)\n        return f\n\n    def errorhandler(self, code):\n        def decorate(f):\n            self.error_handlers[code] = f\n            return f\n        return decorate\n\n    def url_for(self, endpoint, **values):\n        for r in self.rules:\n            if r.endpoint == endpoint:\n                return r.build(dict(values))\n        raise LookupError(f"no endpoint {endpoint!r}")\n\n    @staticmethod\n    def _to_response(rv):\n        status, headers = None, None\n        if isinstance(rv, tuple):\n            if len(rv) == 3:\n                rv, status, headers = rv\n            elif isinstance(rv[1], (dict, list)):\n                rv, headers = rv\n            else:\n                rv, status = rv\n        if isinstance(rv, Response):\n            response = rv\n        elif isinstance(rv, (dict, list)):\n            response = jsonify(rv)\n        elif isinstance(rv, (str, bytes)):\n            response = Response(rv)\n        elif rv is None:\n            raise TypeError("the view function did not return a valid response")\n        else:\n            raise TypeError(f"can\'t turn {type(rv).__name__} into a response")\n        if status is not None:\n            response.status_code = status\n        for k, v in (headers.items() if isinstance(headers, dict) else headers or []):\n            response.headers[k] = v\n        return response\n\n    def _error(self, e):\n        handler = self.error_handlers.get(e.code)\n        if handler:\n            return self._to_response(handler(e))\n        body = f"<h1>{e.code} {e.name}</h1><p>{e.description}</p>"\n        return Response(body, e.code)\n\n    def _dispatch(self, req):\n        path = req.path\n        allowed = set()\n        for r in self.rules:\n            values = r.match(path)\n            if values is None:\n                continue\n            if req.method in r.methods:\n                req.view_args = values\n                return r.view(**values)\n            allowed |= r.methods\n        if allowed:\n            raise _MethodNotAllowed(sorted(allowed | {"OPTIONS"}))\n        abort(404, "The requested URL was not found on the server.")\n\n    def full_dispatch(self, environ):\n        req = Request(environ)\n        token = _current.set((req, _G(), self))\n        try:\n            try:\n                rv = None\n                for f in self.before_funcs:\n                    rv = f()\n                    if rv is not None:\n                        break\n                if rv is None:\n                    rv = self._dispatch(req)\n                response = self._to_response(rv)\n            except _MethodNotAllowed as e:\n                response = self._error(HTTPException(405))\n                response.headers["Allow"] = ", ".join(e.allowed)\n            except HTTPException as e:\n                response = self._error(e)\n            except Exception as e:\n                if self.config.get("TESTING") and 500 not in self.error_handlers:\n                    raise\n                response = self._error(HTTPException(500)) if 500 not in self.error_handlers else self._to_response(self.error_handlers[500](e))\n                response.status_code = 500\n            for f in self.after_funcs:\n                response = f(response) or response\n            if req.method == "HEAD":\n                response.headers["Content-Length"] = len(response.data)\n                response.data = b""\n            return response\n        finally:\n            _current.reset(token)\n\n    def __call__(self, environ, start_response):\n        response = self.full_dispatch(environ)\n        if "Content-Length" not in response.headers:\n            response.headers["Content-Length"] = len(response.data)\n        start_response(response.status, response.headers.items())\n        return [response.data]\n\n    def test_client(self):\n        return _TestClient(self)\n\n\nclass _MethodNotAllowed(Exception):\n    def __init__(self, allowed):\n        self.allowed = allowed\n\n\nclass Flask(_App):\n    pass\n\n\nclass _TestClient:\n    def __init__(self, app):\n        self.app = app\n        self.cookies = {}\n\n    def open(self, path, method="GET", json=None, data=None, headers=None, query_string=None):\n        url = urlsplit(path)\n        query = url.query\n        if query_string:\n            query = (query + "&" if query else "") + urlencode(query_string, doseq=True)\n        body, ctype = b"", None\n        if json is not None:\n            import json as _json\n            body, ctype = _json.dumps(json).encode("utf-8"), "application/json"\n        elif isinstance(data, dict):\n            body, ctype = urlencode(data, doseq=True).encode("utf-8"), "application/x-www-form-urlencoded"\n        elif data is not None:\n            body = data.encode("utf-8") if isinstance(data, str) else data\n        environ = {"REQUEST_METHOD": method.upper(), "PATH_INFO": url.path or "/", "QUERY_STRING": query,\n                   "SERVER_NAME": "localhost", "SERVER_PORT": "80", "wsgi.url_scheme": "http",\n                   "wsgi.input": io.BytesIO(body), "CONTENT_LENGTH": str(len(body))}\n        headers = dict(headers or {})\n        if ctype and not any(k.lower() == "content-type" for k in headers):\n            environ["CONTENT_TYPE"] = ctype\n        for k, v in headers.items():\n            if k.lower() == "content-type":\n                environ["CONTENT_TYPE"] = v\n            else:\n                environ["HTTP_" + k.upper().replace("-", "_")] = v\n        if self.cookies:\n            environ["HTTP_COOKIE"] = "; ".join(f"{k}={v}" for k, v in self.cookies.items())\n        response = self.app.full_dispatch(environ)\n        for k, v in response.headers.items():\n            if k.lower() == "set-cookie":\n                name, _, rest = v.partition("=")\n                self.cookies[name] = rest.split(";")[0]\n        return response\n\n    def get(self, path, **kw):\n        return self.open(path, "GET", **kw)\n\n    def post(self, path, **kw):\n        return self.open(path, "POST", **kw)\n\n    def put(self, path, **kw):\n        return self.open(path, "PUT", **kw)\n\n    def patch(self, path, **kw):\n        return self.open(path, "PATCH", **kw)\n\n    def delete(self, path, **kw):\n        return self.open(path, "DELETE", **kw)\n')

**Predict before you run:** what will this print?



In [ ]:
from miniflask import Flask, request

app = Flask(__name__)

@app.route("/")
def home():
    return "Welcome to the shop"

@app.route("/hello/<name>")
def hello(name):
    times = request.args.get("times", 1, type=int)
    return f"Hello, {name}! " * times

client = app.test_client()
for url in ["/", "/hello/Ada?times=2", "/nowhere"]:
    response = client.get(url)
    print(response.status, repr(response.text[:40]))

In [ ]:
from miniflask import Flask, request, jsonify, abort, g

app = Flask(__name__)
KEYS = {"k-ada": "ada", "k-bob": "bob"}
BOOKS = {1: {"id": 1, "title": "Emma"}}

@app.before_request
def identify():
    g.user = KEYS.get(request.headers.get("X-Api-Key"))
    if request.path.startswith("/api") and g.user is None:
        return jsonify(error="an API key is required"), 401

@app.after_request
def stamp(response):
    response.headers["X-Served-By"] = "atlas-shop"
    return response

@app.errorhandler(404)
def not_found(e):
    return {"error": e.description}, 404

@app.get("/api/books/<int:book_id>")
def get_book(book_id):
    if book_id not in BOOKS:
        abort(404, f"no book {book_id}")
    return {**BOOKS[book_id], "viewer": g.user}

client = app.test_client()
print(client.get("/api/books/1").status, client.get("/api/books/1").json)
r = client.get("/api/books/1", headers={"X-Api-Key": "k-ada"})
print(r.status, r.json, r.headers["X-Served-By"])
print(client.get("/api/books/9", headers={"X-Api-Key": "k-bob"}).json)

> **Key idea.** Flask apps register view functions with @app.route(rule, methods=...) (or @app.get, @app.post ...); converters (<int:id>, <path:p>) turn path parts into arguments, and non-matching paths are 404s. Views return a str, a dict or list (JSON), a Response, or a tuple with a status and headers; never None. flask.request gives method, path, headers, args (with get(..., type=int)), form, get_json() and cookies; jsonify, redirect, make_response and url_for build responses and URLs. abort(code, description) stops a view; errorhandler replaces error pages; before_request (which may answer early), after_request and g share per-request work. Application factories and blueprints structure larger apps, and test_client with pytest fixtures tests them without a server.

### Exercise: A first Flask app

Write `create_app()`, returning a Flask app with three routes: `GET /` returns the text `Welcome to Atlas Books`; `GET /hello/<name>` returns `Hello, NAME!`, in capitals if the query string has `shout=yes`; and `GET /square/<int:n>` returns JSON `{"n": N, "square": N*N}`.

In [ ]:
from miniflask import Flask, request

def create_app():
    app = Flask(__name__)
    ___
    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hello():\n    \"routes, converters and query strings\"\n    client = create_app().test_client()\n    assert client.get(\"/\").text == \"Welcome to Atlas Books\"\n    assert client.get(\"/hello/Ada\").text == \"Hello, Ada!\" and client.get(\"/hello/Ada?shout=yes\").text == \"HELLO, ADA!\"\n    r = client.get(\"/square/12\")\n    assert r.status_code == 200 and r.get_json() == {\"n\": 12, \"square\": 144} and r.headers[\"Content-Type\"] == \"application/json\"\n    assert client.get(\"/square/twelve\").status_code == 404 and client.post(\"/\").status_code == 405")

### Exercise: A JSON API

Write `create_app()` with an in-memory dict of notes (`{id: {"id": id, "text": ...}}`, starting empty, ids counting from 1) and these routes: `GET /notes` (a list, by id; with `?q=word`, only notes whose text contains it, any case), `POST /notes` (JSON `{"text": ...}`; `201` with the note and a `Location` header from `url_for`; `422` with `{"error": "text is required"}` if the text is missing or blank), `GET /notes/<int:note_id>` (the note, or `404` via `abort`), and `DELETE /notes/<int:note_id>` (`204` with an empty body, or `404`). Add a `404` error handler that returns `{"error": DESCRIPTION}`.

In [ ]:
from miniflask import Flask, request, abort, url_for

def create_app():
    app = Flask(__name__)
    notes = {}
    ___
    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_crud():\n    \"status codes, Location and errors\"\n    client = create_app().test_client()\n    r = client.post(\"/notes\", json={\"text\": \"Buy tea\"})\n    assert r.status_code == 201 and r.get_json() == {\"id\": 1, \"text\": \"Buy tea\"} and r.headers[\"Location\"] == \"/notes/1\"\n    client.post(\"/notes\", json={\"text\": \"call Bob about TEA\"})\n    client.post(\"/notes\", json={\"text\": \"water plants\"})\n    assert [n[\"id\"] for n in client.get(\"/notes\").get_json()] == [1, 2, 3]\n    assert [n[\"id\"] for n in client.get(\"/notes?q=tea\").get_json()] == [1, 2]\n    assert client.post(\"/notes\", json={\"text\": \"  \"}).status_code == 422 and client.post(\"/notes\", json={}).get_json() == {\"error\": \"text is required\"}\n    assert client.get(\"/notes/2\").get_json()[\"text\"] == \"call Bob about TEA\"\n    r = client.delete(\"/notes/2\")\n    assert r.status_code == 204 and r.data == b\"\"\n    assert client.get(\"/notes/2\").status_code == 404 and \"error\" in client.get(\"/notes/2\").get_json()\n    assert client.post(\"/notes\", data=\"not json\", headers={\"Content-Type\": \"text/plain\"}).status_code == 415")

### Exercise: JSON errors everywhere

An API should never answer with an HTML error page. Write `json_errors(app)`: register error handlers on `app` for `400`, `404`, `405`, `415`, `422` and `500` that all return JSON `{"error": NAME, "detail": DESCRIPTION}` with the right status (`NAME` is the exception's `name`, such as `"Not Found"`), and for `500` the detail `"something went wrong"` (never the exception's message). Return the app. The checks use the given `make_app`.

In [ ]:
from miniflask import Flask, request, abort

def make_app():
    app = Flask(__name__)

    @app.post("/divide")
    def divide():
        data = request.get_json()
        if "b" not in data:
            abort(422, "b is required")
        return {"result": data["a"] / data["b"]}

    return app

def json_errors(app):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_errors():\n    \"error handlers\"\n    client = json_errors(make_app()).test_client()\n    assert client.post(\"/divide\", json={\"a\": 6, \"b\": 3}).get_json() == {\"result\": 2.0}\n    r = client.post(\"/divide\", json={\"a\": 6})\n    assert r.status_code == 422 and r.get_json()[\"detail\"] == \"b is required\"\n    assert r.get_json()[\"error\"] in (\"Unprocessable Content\", \"Unprocessable Entity\")\n    r = client.post(\"/divide\", json={\"a\": 6, \"b\": 0})\n    assert r.status_code == 500 and r.get_json()[\"detail\"] == \"something went wrong\" and \"division\" not in r.text\n    assert client.get(\"/divide\").status_code == 405 and client.get(\"/divide\").get_json()[\"error\"] == \"Method Not Allowed\"\n    assert client.get(\"/nope\").get_json()[\"error\"] == \"Not Found\"\n    assert client.post(\"/divide\", data=\"x\", headers={\"Content-Type\": \"text/plain\"}).status_code == 415\n    assert client.post(\"/divide\", data=\"{bad\", headers={\"Content-Type\": \"application/json\"}).get_json()[\"error\"] == \"Bad Request\"")

### Exercise: Keys, g and an extra header

Write `create_app(keys)`: `keys` maps API keys to user names. A `before_request` hook reads the `X-Api-Key` header, sets `g.user` (or `None`), and answers `401` with `{"error": "unauthorised"}` for any path under `/api/` without a valid key. An `after_request` hook adds `X-Request-Count`: how many requests this app has handled so far, counting this one. Routes: `GET /` (public, returning `{"hello": USER}`, with `null` when anonymous) and `GET /api/me` (returning `{"user": USER}`).

In [ ]:
from miniflask import Flask, request, g

def create_app(keys):
    app = Flask(__name__)
    ___
    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hooks():\n    \"before_request, after_request and g\"\n    client = create_app({\"k1\": \"ada\"}).test_client()\n    r = client.get(\"/\")\n    assert r.get_json() == {\"hello\": None} and r.headers[\"X-Request-Count\"] == \"1\"\n    assert client.get(\"/\", headers={\"X-Api-Key\": \"k1\"}).get_json() == {\"hello\": \"ada\"}\n    r = client.get(\"/api/me\")\n    assert r.status_code == 401 and r.get_json() == {\"error\": \"unauthorised\"} and r.headers[\"X-Request-Count\"] == \"3\"\n    assert client.get(\"/api/me\", headers={\"X-Api-Key\": \"k1\"}).get_json() == {\"user\": \"ada\"}\n    assert client.get(\"/api/me\", headers={\"X-Api-Key\": \"nope\"}).status_code == 401")

### Exercise: A blueprint and a factory

Write a blueprint `books` (with `url_prefix="/api/v1/books"`) with `GET ""` (the list) and `GET "/<int:book_id>"` (one book, or `404` JSON `{"error": "no book N"}`), reading the books from `current_books()`, which returns the list in the app's config under `"BOOKS"` (given). Each book in the list response gains a `"url"` built with `url_for("books.get_book", ...)`. Then `create_app(config=None)`: an app whose config is updated from `config`, with the blueprint registered.

In [ ]:
from miniflask import Flask, Blueprint, abort, url_for

books = Blueprint("books", __name__, url_prefix="/api/v1/books")
APP = {}

def current_books():
    return APP["app"].config.get("BOOKS", [])

___

def create_app(config=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_blueprints():\n    \"blueprints, prefixes and url_for\"\n    data = [{\"id\": 1, \"title\": \"Emma\"}, {\"id\": 2, \"title\": \"Dune\"}]\n    client = create_app({\"BOOKS\": data}).test_client()\n    r = client.get(\"/api/v1/books\")\n    assert r.get_json() == [{\"id\": 1, \"title\": \"Emma\", \"url\": \"/api/v1/books/1\"}, {\"id\": 2, \"title\": \"Dune\", \"url\": \"/api/v1/books/2\"}]\n    assert client.get(\"/api/v1/books/2\").get_json() == {\"id\": 2, \"title\": \"Dune\"}\n    r = client.get(\"/api/v1/books/7\")\n    assert r.status_code == 404 and r.get_json() == {\"error\": \"no book 7\"}\n    other = create_app({\"BOOKS\": []}).test_client()\n    assert other.get(\"/api/v1/books\").get_json() == []")

### Exercise: A form, and a redirect

Write `create_app()` for a sign-up form: `POST /signup` with form fields `email` and `name`: if the email has no `@` or the name is blank, answer `400` with the text `Please give a name and a valid email`; otherwise add `(name, email lower-cased)` to `app.config["SIGNUPS"]` (a list, starting empty) and **redirect** with `303` to `url_for("thanks", name=NAME)`. `GET /thanks` returns `Thanks, NAME!`, taking the name from the query string.

In [ ]:
from miniflask import Flask, request, redirect, url_for

def create_app():
    app = Flask(__name__)
    app.config["SIGNUPS"] = []
    ___
    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_forms():\n    \"forms and Post/Redirect/Get\"\n    app = create_app()\n    client = app.test_client()\n    r = client.post(\"/signup\", data={\"email\": \"Ada@Example.com\", \"name\": \"Ada\"})\n    assert r.status_code == 303 and r.headers[\"Location\"] == \"/thanks?name=Ada\"\n    assert client.get(r.headers[\"Location\"]).text == \"Thanks, Ada!\"\n    assert app.config[\"SIGNUPS\"] == [(\"Ada\", \"ada@example.com\")]\n    bad = client.post(\"/signup\", data={\"email\": \"nope\", \"name\": \"Bob\"})\n    assert bad.status_code == 400 and bad.text == \"Please give a name and a valid email\" and len(app.config[\"SIGNUPS\"]) == 1")

### Exercise: Testing a Flask app

`shop.py` (in the exercise) has a Flask `create_app()` with a small basket API: `POST /basket` adds `{"item": ..., "qty": ...}` (qty a positive whole number, default 1), `GET /basket` lists the basket with a total count, and `DELETE /basket/<item>` removes an item (`404` if it isn't there). Write pytest tests with a `client` fixture that makes a **fresh app** for each test, which pass on the correct code and catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('shop.py', 'w', encoding='utf-8') as f:
    f.write('from miniflask import Flask, request\n\ndef create_app():\n    app = Flask(__name__)\n    basket = {}\n\n    @app.post("/basket")\n    def add():\n        data = request.get_json(silent=True) or {}\n        item, qty = data.get("item"), data.get("qty", 1)\n        if not item or not isinstance(qty, int) or qty < 1:\n            return {"error": "an item and a positive qty are needed"}, 422\n        basket[item] = basket.get(item, 0) + qty\n        return {"item": item, "qty": basket[item]}, 201\n\n    @app.get("/basket")\n    def show():\n        return {"items": dict(sorted(basket.items())), "count": sum(basket.values())}\n\n    @app.delete("/basket/<item>")\n    def remove(item):\n        if item not in basket:\n            return {"error": f"no {item} in the basket"}, 404\n        del basket[item]\n        return "", 204\n\n    return app\n')

In [ ]:
import pytest
from shop import create_app

@pytest.fixture
def client():
    ___

def test_add(client):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"shop.py\").read()\nBROKEN = {\n    \"shared basket\": GOOD.replace(\"    app = Flask(__name__)\\n    basket = {}\\n\", \"    app = Flask(__name__)\\n\").replace(\"def create_app():\", \"basket = {}\\n\\ndef create_app():\"),\n    \"adding replaces\": GOOD.replace(\"basket[item] = basket.get(item, 0) + qty\", \"basket[item] = qty\"),\n    \"zero allowed\": GOOD.replace(\"qty < 1\", \"qty < 0\"),\n    \"wrong status\": GOOD.replace('return {\"item\": item, \"qty\": basket[item]}, 201', 'return {\"item\": item, \"qty\": basket[item]}'),\n    \"count of kinds\": GOOD.replace('\"count\": sum(basket.values())', '\"count\": len(basket)'),\n    \"missing delete is fine\": GOOD.replace('return {\"error\": f\"no {item} in the basket\"}, 404', 'return \"\", 204'),\n}\ndef test_catches():\n    \"your tests pass on the correct app and catch all six broken versions\"\n    results = against(source, \"shop\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"shop.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Bug hunt: A view with nothing to return

The view builds its answer but forgets to **return** it, so it returns `None`, which Flask can't turn into a response, and the request fails with a `TypeError`. Return the result, so this prints `200 {'total': 6}`.

In [ ]:
from miniflask import Flask

app = Flask(__name__)

@app.get("/total/<int:a>/<int:b>")
def total(a, b):
    result = {"total": a + b}

r = app.test_client().get("/total/2/4")
print(r.status_code, r.get_json())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"return the response\"\n    expect_output(\"200 {'total': 6}\")")

### Bug hunt: A method that wasn't allowed

The route only accepts `GET` (the default), so the form's `POST` is refused with `405` and this prints `405 Method Not Allowed`. Allow `POST` on it, so this prints `201 Created`.

In [ ]:
from miniflask import Flask, request

app = Flask(__name__)
messages = []

@app.route("/messages")
def add_message():
    messages.append(request.get_json()["text"])
    return {"count": len(messages)}, 201

r = app.test_client().post("/messages", json={"text": "hi"})
print(r.status)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the right methods\"\n    expect_output(\"201 Created\")")

### Bug hunt: An id that stayed text

The rule uses `<book_id>`, so `book_id` arrives as the **string** `"1"`, and the dict's keys are integers, so every lookup fails and this prints `404`. Use the `int` converter, so it prints `200 Emma`.

In [ ]:
from miniflask import Flask, abort

app = Flask(__name__)
BOOKS = {1: "Emma", 2: "Dune"}

@app.get("/books/<book_id>")
def book(book_id):
    if book_id not in BOOKS:
        abort(404)
    return BOOKS[book_id]

r = app.test_client().get("/books/1")
print(r.status_code, r.text if r.status_code == 200 else "")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"converters\"\n    assert output.strip() == \"200 Emma\", output")

### Puzzle: A Flask endpoint

Arrange the lines of a Flask route that creates a book from JSON and answers `201` with its URL. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the status Flask gives for /books/abc when the rule is /books/<int:book_id>
q = ___   # what a view returning ({"a": 1}, 201) sends as its Content-Type
r = ___   # request.args.get("page", 1, type=int) for the query string page=two
s = ___   # the decorator for code that runs before every view
t = ___   # url_for("api.book", book_id=7) when the blueprint "api" has url_prefix "/api" and the rule is "/books/<int:book_id>"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 404 and q == \"application/json\" and r == 1, \"p, q and r\"\n    assert s in (\"before_request\", \"@app.before_request\") and t == \"/api/books/7\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a paginated, filtered catalogue

Write `create_app(books)` serving `books` (a list of dicts with `id`, `title`, `author` and `year`) at `GET /books`, with query parameters: `author` (exact match, any case), `since` (books with `year` at or after it), `sort` (`title`, `year` or `-year`; default `id`), `page` (from 1) and `per_page` (1 to 50, default 10). Invalid numbers, a `per_page` out of range, or an unknown `sort` answer `400` with `{"error": "..."}` naming the parameter (such as `"invalid page"`). The response is `{"items": [...], "page": P, "per_page": N, "total": T, "pages": max(1, ceil(T / N))}`, plus a **`Link`** header with `<URL>; rel="next"` and `<URL>; rel="prev"` parts (joined with `", "`) where those pages exist, each URL keeping the other query parameters (built with `url_for("list_books", ...)`). A `page` past the last gives an empty `items` list, not an error.

In [ ]:
import math
from miniflask import Flask, request, url_for

def create_app(books):
    app = Flask(__name__)
    ___
    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_catalogue():\n    \"filtering, sorting, paging and links\"\n    books = [{\"id\": i, \"title\": t, \"author\": a, \"year\": y} for i, (t, a, y) in enumerate([\n        (\"Emma\", \"Austen\", 1815), (\"Persuasion\", \"Austen\", 1817), (\"Dracula\", \"Stoker\", 1897), (\"Middlemarch\", \"Eliot\", 1871),\n        (\"Ivanhoe\", \"Scott\", 1819), (\"Cranford\", \"Gaskell\", 1853), (\"Sanditon\", \"Austen\", 1925)], start=1)]\n    client = create_app(books).test_client()\n    r = client.get(\"/books?per_page=3\")\n    data = r.get_json()\n    assert [b[\"id\"] for b in data[\"items\"]] == [1, 2, 3] and (data[\"total\"], data[\"pages\"], data[\"page\"]) == (7, 3, 1)\n    assert r.headers[\"Link\"] == '</books?per_page=3&page=2>; rel=\"next\"'\n    r = client.get(\"/books?author=austen&sort=-year&per_page=2&page=2\")\n    data = r.get_json()\n    assert [b[\"title\"] for b in data[\"items\"]] == [\"Emma\"] and data[\"pages\"] == 2\n    assert r.headers[\"Link\"] == '</books?author=austen&sort=-year&per_page=2&page=1>; rel=\"prev\"'\n    r = client.get(\"/books?since=1850&sort=title&per_page=2&page=2\")\n    assert [b[\"title\"] for b in r.get_json()[\"items\"]] == [\"Middlemarch\", \"Sanditon\"]\n    assert r.headers[\"Link\"] == '</books?since=1850&sort=title&per_page=2&page=1>; rel=\"prev\"'\n    r = client.get(\"/books?per_page=2&page=2\")\n    assert 'rel=\"next\"' in r.headers[\"Link\"] and 'rel=\"prev\"' in r.headers[\"Link\"]\n    assert client.get(\"/books?page=9\").get_json()[\"items\"] == [] and \"Link\" not in client.get(\"/books\").headers\n    for query, word in [(\"page=zero\", \"page\"), (\"per_page=0\", \"per_page\"), (\"per_page=51\", \"per_page\"), (\"since=x\", \"since\"), (\"sort=price\", \"sort\")]:\n        r = client.get(f\"/books?{query}\")\n        assert r.status_code == 400 and word in r.get_json()[\"error\"], query")

## Warm-up

That's a complete Flask application: an `app`, two **view functions** registered with `@app.route`, and a test client that calls them without a network. Flask did the parsing, the routing, the `404` and the status lines: everything you built by hand last week. (On your own computer, `from flask import Flask, request` gives the real one, and the same code runs unchanged.)

## Routes and views

```python
from flask import Flask

app = Flask(__name__)

@app.route("/books/<int:book_id>", methods=["GET", "DELETE"])
def book(book_id):
    ...
```

- `@app.route(rule, methods=[...])` registers a **view function** (also called an endpoint) for a URL rule; `methods` defaults to `["GET"]`. Flask 2 added shortcuts: `@app.get(rule)`, `@app.post(rule)`, `@app.put`, `@app.patch`, `@app.delete`.
- **Converters** in the rule turn path parts into arguments: `<name>` (any text without `/`), `<int:book_id>`, `<float:x>`, `<path:rest>` (slashes allowed). A path that doesn't fit, such as `/books/abc` for an `int`, is simply a `404`.
- A view **returns** the response: a `str` (sent as HTML), a `dict` or `list` (sent as JSON), a `Response` object, or a **tuple**: `(body, status)`, `(body, headers)` or `(body, status, headers)`. Returning `None` is an error: the classic forgotten `return`.
- Flask answers `405 Method Not Allowed` (with `Allow`) for a known path with the wrong method, and `HEAD` and `OPTIONS` automatically.

## Requests and responses

The current request is available as **`flask.request`**, a "context-local" object: it looks like a global, but each request (and each thread) sees its own.

- `request.method`, `request.path`, `request.headers` (case-insensitive), `request.cookies`.
- `request.args`: the query string, as a dict of first values: `request.args.get("page", 1, type=int)` converts, falling back to the default on bad input; `.getlist("tag")` gives every value.
- `request.form`: an HTML form's fields; `request.get_json()`: the JSON body (Flask refuses a body without `Content-Type: application/json` with `415`, and invalid JSON with `400`; `get_json(silent=True)` returns `None` instead).
- `jsonify(...)` makes a JSON response explicitly; `redirect(url, code=302)` a redirect (use `303` after a form); `make_response(body, status, headers)` a `Response` you can change before returning (adding cookies with `set_cookie`).
- **`url_for(endpoint, **values)`** builds a URL from a view's name and its parameters, so URLs are written in one place: `url_for("book", book_id=7)` gives `/books/7`.

## Errors and hooks

- **`abort(404)`** (or any code) stops the view at once and sends that error. `abort(422, "title is required")` adds a description.
- **`@app.errorhandler(404)`** replaces Flask's default error page: an API returns JSON, such as `{"error": e.description}, 404`. A `500` handler catches unexpected exceptions (with the details logged, not sent).
- **`@app.before_request`** functions run before every view; if one returns a response, the view isn't called at all, which suits authentication checks. **`@app.after_request`** functions receive the response and return it (perhaps with headers added) after every view.
- **`g`** holds data for the current request only: a `before_request` function can set `g.user`, and views read it.

## Factories, blueprints and tests

As an app grows, two patterns keep it manageable:

- An **application factory**, `create_app(config=None)`, builds and returns a new app, applying the configuration it's given. Tests can then make a fresh app with test settings (an in-memory database, `TESTING = True`) for each test, and the real app reads its settings from the environment (Week 79).
- **Blueprints** group related routes into modules: `api = Blueprint("api", __name__, url_prefix="/api/v1")`, routes registered with `@api.get(...)`, and `app.register_blueprint(api)` in the factory. Endpoint names gain the blueprint's name: `url_for("api.book", book_id=7)`.

Testing uses the **test client** with pytest fixtures (Week 72):

```python
import pytest
from shop import create_app

@pytest.fixture
def client():
    app = create_app({"TESTING": True})
    return app.test_client()

def test_missing_book(client):
    response = client.get("/api/v1/books/999")
    assert response.status_code == 404 and response.get_json() == {"error": "no book 999"}
```

`client.get`, `.post(json=...)`, `.post(data={...})` (a form), `.put`, `.delete`, each with optional `headers=` and `query_string=`, return a response with `status_code`, `headers`, `data`, `text` and `get_json()`.

## Practice

These exercises use the page's `miniflask.py`; on your own computer, change the import to `from flask import ...` and the same code runs.

## On your own computer

Install the real Flask and run one of today's apps:

```bash
pip install flask
```

```bash
flask --app hello run --debug
```

(with your code in `hello.py`, where `app` is defined or `create_app` is the factory). `--debug` reloads on every change and shows an interactive traceback in the browser for errors: never use it in production. For HTML pages, Flask uses **Jinja** templates: `render_template("book.html", book=book)` fills `templates/book.html`, escaping values automatically so user input can't inject HTML.

## Quiz

## Daily challenge

## Before you go

Run today's notes API with the real Flask on your own computer, and call it with `curl` and your Week 76 client. Tomorrow: **FastAPI**, a newer framework that reads your type hints to validate requests, convert parameters and write the API's documentation for you.